In [ ]:
%run Include.ipynb
%run Net.ipynb
%run Data.ipynb
%run Topo_treatment.ipynb
%run Viewer.ipynb

import pandas as pd
from datetime import datetime
from tqdm.notebook import tqdm
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import os


class GAN(object):
    
    def __init__(self, general, adv_params, G_arch, D_arch):
        
        lr        = general["learning_rate"]
        beta1     = general["beta1"]
        beta2     = general["beta2"]
        loss_mode = general["loss"]
        reduction = general["reduction"]
        
        self.N_critic = adv_params["wgangp"]["N_CRITIC"]
        
        cudnn.benchmark = FLAGS.cudnn_benchmark
        gpu_num     = FLAGS.gpu_num
        self.device = torch.device("cuda:0" if torch.cuda.is_available()
                      and FLAGS.gpu_enable else "cpu")
        torch.manual_seed(random.randint(1, 10000))
        
        self.inputG_dims, G_layers = Net.parse_layers(G_arch)
        self.inputD_dims, D_layers = Net.parse_layers(D_arch)
        self.netG = Network_template(gpu_num, G_layers).to(self.device)
        Net.init_weights(self.netG, "normal")
        self.netD = Network_template(gpu_num, D_layers).to(self.device)
        Net.init_weights(self.netD, "normal")
        
        self.et         = Edges_(adv_params, debug=False)
        self.criterion  = GANLoss(loss_mode, reduction).to(self.device)
        self.criterionT = GANLoss("vanilla_topo", "sum").to(self.device)
        self.optimizerD = optim.Adam(self.netD.parameters(), lr=lr, betas=(beta1, beta2))
        #self.optimizerD = optim.Adam(filter(lambda p: p.requires_grad, self.netD.parameters()), lr=lr, betas=(beta1,beta2))
        self.optimizerG = optim.Adam(self.netG.parameters(), lr=lr, betas=(beta1, beta2))
        
    def save_noise_(self, shape, name):
        '''
        shape: shape of the noise, usually it is [batch_size, 128, 1, 1]
        name: should be like 128_128_1_1_0.dat
        all noise are saved under D:/Data/fixed_z/
        '''
        z_ = torch.randn(shape, device=self.device)
        FileIO.write_binary('D:/Data/fixed_z/'+name, z_.cpu().numpy().flatten(), list(z_.shape), 'f')
        
    def sample_(self, shape):
        z_ = torch.randn(shape, device=self.device)
        return self.netG(z_)
    
    def sample_save_(self, name, shape, direc, scalor, offset):
        '''
        shape: shape of the noise, usually it is [batch_size, 128, 1, 1]
        name: should be like 128_128_1_1_0.dat
        all noise are saved under D:/Data/fixed_z/
        '''
        Path(direc).mkdir(parents=True, exist_ok=True)
        if FLAGS.continue_model:
            self.netG.load_state_dict(torch.load('%s/netG_step_%d.pth' % (FLAGS.model_save, FLAGS.model_step)))
            self.netD.load_state_dict(torch.load('%s/netD_step_%d.pth' % (FLAGS.model_save, FLAGS.model_step)))
            print("Models loaded at step %d" % FLAGS.model_step)
        
        i = 0
        interval = 1000
        z_ = FileIO.read_binary('D:/Data/fixed_z/'+name, shape, 'f')
        while True:
            si = i
            se = np.min((si + interval, z_.shape[0]))
            z_sub_ = z_[si:se,:]
            z_sub_ = torch.from_numpy(z_sub_).to(self.device)
            samples = self.netG(z_sub_)
            FileIO.save_image_batch(samples.detach().cpu().numpy(), direc, 'gen', scalor, si)
            i = se
            if i >= z_.shape[0]:
                break
     
    def D_iteration(self, Dreal_device, Dfake_device):
        self.netD.zero_grad()      
        errD = self.criterion(["D", self.netD, self.device, Dreal_device, Dfake_device])
        errD.backward()
        self.optimizerD.step()
        return errD.item()
    
    def G_iteration(self, Dfake_device, withTopo, epoch_index: int):
        # Dfake_device - сгенерированные фото 
        self.netG.zero_grad()
        errG = self.criterion(["G", self.netD, Dfake_device])
        errG.backward(retain_graph=withTopo)
   
        if withTopo:
            tp_wgt   = self.et.return_tp_weight()
            # переслать фото с видеокарточки на компьютер и применить функцию fix_with_topo
            fake_fix, mean_wasdis = self.et.fix_with_topo(Dfake_device.detach().cpu().numpy(),
                                    self.et.return_target_dim(), -1.0, 1.0, blind=self.et.blind(), epoch_index=epoch_index)
            fake_fix = torch.from_numpy(fake_fix).to(self.device)
            errT     = self.criterionT([Dfake_device, fake_fix]) * tp_wgt
            errT.backward()
            self.optimizerG.step()
            return [errG.item(), errT.item(), mean_wasdis]
        self.optimizerG.step()
        return errG.item()

#     def train(self, data_params, withTopo):       
#         epochs        = data_params["epochs"]
#         batch_size    = data_params["batch_size"]
#         batch_workers = data_params["batch_workers"]
#         shuffle       = data_params["shuffle"]
#         drop_last     = data_params["drop_last"]
#         dataloader    = Data_fetcher.fetch_dataset(FLAGS.dataset, batch_size, batch_workers, shuffle, drop_last, 0.5)
#         log           = open(FLAGS.log_path, "a")
        
#         step = 0
#         if withTopo:
#             self.et.load_pd_pool(FLAGS.pds_path, "dat", 1.0, batch_size)
#         if FLAGS.continue_model:
#             self.netG.load_state_dict(torch.load('%s/netG_step_%d.pth' % (FLAGS.model_save, FLAGS.model_step)))
#             self.netD.load_state_dict(torch.load('%s/netD_step_%d.pth' % (FLAGS.model_save, FLAGS.model_step)))
#             step = FLAGS.model_step + 1
        
#         g_lrec = []
#         d_lrec = []
#         #fixed_z_ = FileIO.read_binary('D:/Data/fixed_z/128_128_1_1_0.dat', [batch_size]+self.inputG_dims, 'f')
#         #fixed_z_ = torch.from_numpy(fixed_z_).to(self.device)
#         fixed_z_ = torch.randn([128,128,1,1], device=self.device)
#         for epoch in range(epochs):
#             for i, data in enumerate(dataloader, 0):               
#                 Dreal_device = data['image'].to(self.device)
#                 Dfake_device = self.sample_([data['image'].shape[0]]+self.inputG_dims)
#                 d_lrec.append(self.D_iteration(Dreal_device, Dfake_device))
                
#                 if step % self.N_critic == 0:
#                     Dfake_device = self.sample_([data['image'].shape[0]]+self.inputG_dims)
#                     g_lrec.append(self.G_iteration(Dfake_device, withTopo))
#                 step = step + 1
                
#                 if step % FLAGS.print_step == 0:
#                     if withTopo:
#                         msg = ('[%d/%d][%d/%d] D_loss: %.4f G_loss: %.4f T_loss: %.4f Wasserstein_dist: %.4f Step: %d'
#                           %(epoch, epochs, i, len(dataloader), np.mean(np.asarray(d_lrec)), np.mean(np.asarray(g_lrec)[:,0]),
#                             np.mean(np.asarray(g_lrec)[:,1]), np.mean(np.asarray(g_lrec)[:,2]), step))
#                     else:        
#                         msg = ('[%d/%d][%d/%d] D_loss: %.4f G_loss: %.4f Step: %d'
#                           %(epoch, epochs, i, len(dataloader), np.mean(np.asarray(d_lrec)), np.mean(np.asarray(g_lrec)), step))
#                     g_lrec[:] = []
#                     d_lrec[:] = []
#                     print(msg)
#                     log.write(msg+"\n")
#                     log.flush()
#                 if step % FLAGS.save_step == 0:
#                     # ===== Save images ====
#                     Dfake_device_ = self.netG(fixed_z_)
#                     vutils.save_image(Dfake_device_.detach().cpu(),
#                     '%s/generated_step_%d.png' % (FLAGS.image_save, step), normalize=True)
#                     if withTopo:
#                         fix_, _ = self.et.fix_with_topo(Dfake_device_.detach().cpu().numpy(),
#                                 self.et.return_target_dim(), -1.0, 1.0, blind=self.et.blind())
#                         fix_    = torch.from_numpy(np.expand_dims(fix_, axis=1))
#                         vutils.save_image(fix_,
#                         '%s/topo_step_%d.png' % (FLAGS.image_save, step), normalize=True)
#                     # ===== Save models ====
#                     torch.save(self.netG.state_dict(), '%s/netG_step_%d.pth' % (FLAGS.model_save, step))
#                     torch.save(self.netD.state_dict(), '%s/netD_step_%d.pth' % (FLAGS.model_save, step))

#         log.close()
#         print("Training complete.")

    def train(self, data_params, withTopo):       
        epochs        = data_params["epochs"]
        batch_size    = data_params["batch_size"]
        batch_workers = data_params["batch_workers"]
        shuffle       = data_params["shuffle"]
        drop_last     = data_params["drop_last"]
        dataloader    = Data_fetcher.fetch_dataset(FLAGS.dataset, batch_size, batch_workers, shuffle, drop_last, 0.5)
        log           = open(FLAGS.log_path, "a")
        
        step = 0
        if withTopo:
            self.et.load_pd_pool(FLAGS.pds_path, "dat", 1.0, batch_size)
        if FLAGS.continue_model:
            self.netG.load_state_dict(torch.load('%s/netG_step_%d.pth' % (FLAGS.model_save, FLAGS.model_step)))
            self.netD.load_state_dict(torch.load('%s/netD_step_%d.pth' % (FLAGS.model_save, FLAGS.model_step)))
            step = FLAGS.model_step + 1
        
        g_lrec = []
        d_lrec = []
        #fixed_z_ = FileIO.read_binary('D:/Data/fixed_z/128_128_1_1_0.dat', [batch_size]+self.inputG_dims, 'f')
        #fixed_z_ = torch.from_numpy(fixed_z_).to(self.device)
        # my adjustment
        fixed_z_ = torch.randn([batch_size, 128, 1, 1], device=self.device)
        print(f"RANDOM NORMAL IS INITIALIZED FOR N: {batch_size}")
        
        self.csv_path = "C:\\TopoGAN-ECCV2020\\logs\\metric_log_" + datetime.now().strftime("%Y%m%d_%H%M%S") + ".csv"
        csv_log = open(self.csv_path, "w")
        csv_log.write("step,epoch,batch_idx,D_loss,G_loss,T_loss,Wasserstein_dist\n")
        
        
        print(f"Length of dataloader: {len(dataloader)}")
        for epoch in range(epochs):
#             if epoch == 0:
#                 print("Skipping 0 epoch")
                
#                 for i, data in enumerate(dataloader, 0):
#                     pass
#                 continue
                
            for i, data in enumerate(tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}", leave=True), 0):          
                
#                 if i < 21:
#                     print("Skip index:", i)
#                     continue
                    
                Dreal_device = data['image'].to(self.device)
                Dfake_device = self.sample_([data['image'].shape[0]]+self.inputG_dims)
                err_d = self.D_iteration(Dreal_device, Dfake_device)
                d_lrec.append(err_d)
                
                if step % self.N_critic == 0:
                    Dfake_device = self.sample_([data['image'].shape[0]]+self.inputG_dims)
                    g_lrec.append(self.G_iteration(Dfake_device, withTopo, epoch_index=epoch))
                step = step + 1
                
                if (step - 1) % FLAGS.print_step == 0:
                    if withTopo:
                        d_loss = np.mean(np.asarray(d_lrec))
                        try:
                            g_loss = np.mean(np.asarray(g_lrec)[:,0])
                        except IndexError as e:
                            print("ERROR: make sure, that N_critic % print_step == 0")
                            raise e
                        
                        t_loss = np.mean(np.asarray(g_lrec)[:,1])
                        wass_dist = np.mean(np.asarray(g_lrec)[:,2])
                        
                        msg = ('[%d/%d][%d/%d] D_loss: %.4f G_loss: %.4f T_loss: %.4f Wasserstein_dist: %.4f Step: %d'
                               % (epoch, epochs, i, len(dataloader), d_loss, g_loss, t_loss, wass_dist, step))

                    else:
                        msg = ('[%d/%d][%d/%d] D_loss: %.4f G_loss: %.4f Step: %d'
                          %(epoch, epochs, i, len(dataloader), np.mean(np.asarray(d_lrec)), np.mean(np.asarray(g_lrec)), step))
                    g_lrec[:] = []
                    d_lrec[:] = []
                    print(msg)
                    log.write(msg+"\n")
                    log.flush()
                    
                    # New: Save all raw losses to CSV
                    csv_entry = f"{step},{epoch},{i},{d_loss},{g_loss},{t_loss},{wass_dist}"
#                     if withTopo and step % self.N_critic == 0:
#                         csv_entry += f"{g_loss},{t_loss},{wass_dist}"
#                     elif not withTopo and step % self.N_critic == 0:
#                         csv_entry += f"{g_losses},,"
#                     else:
#                         csv_entry += ",,,"
                    csv_log.write(csv_entry + "\n")
                    csv_log.flush()
                    
                if step % FLAGS.save_step == 0:
                    # ===== Save images ====
                    print("Saving image...")
                    Dfake_device_ = self.netG(fixed_z_)
                    vutils.save_image(Dfake_device_.detach().cpu(),
                    '%s/gen_step_%d.png' % (FLAGS.image_save, step), normalize=True)
                    if withTopo:
                        fix_, _ = self.et.fix_with_topo(Dfake_device_.detach().cpu().numpy(),
                                self.et.return_target_dim(), -1.0, 1.0, blind=self.et.blind())
                        fix_    = torch.from_numpy(np.expand_dims(fix_, axis=1))
                        vutils.save_image(fix_,
                        '%s/top_step_%d.png' % (FLAGS.image_save, step), normalize=True)
                    # ===== Save models ====
                    torch.save(self.netG.state_dict(), '%s/netG_step_%d.pth' % (FLAGS.model_save, step))
                    torch.save(self.netD.state_dict(), '%s/netD_step_%d.pth' % (FLAGS.model_save, step))

        log.close()
        print("Training complete.")
        
        
#     def plot_losses(self):
#         """Plot training losses from CSV log file"""
#         df = pd.read_csv(self.csv_path)
        
#         plt.figure(figsize=(12, 6))
        
#         # Plot D loss
#         plt.plot(df['step'], df['D_loss'], label='Discriminator Loss', alpha=0.7)
        
#         plt.plot(df['step'], df['G_loss'], label='Generator Loss', alpha=0.7)
            
#         # Plot topological losses if available
#         plt.plot(df['step'], df['T_loss'], label='Topological Loss', alpha=0.7)
            
# #         if 'Wasserstein_dist' in df.columns and not df['Wasserstein_dist'].isnull().all():
#         plt.plot(df['step'], df['Wasserstein_dist'], label='Wasserstein Distance', alpha=0.7)

#         plt.title('Training Loss Progression')
#         plt.xlabel('Training Step')
#         plt.ylabel('Loss Value')
#         plt.legend()
#         plt.grid(True)
#         plt.tight_layout()
#         plt.show()



    def plot_losses(self, date_str, save_path, logscale: dict = {}):
        """
        Overlay four losses each on its own colored y-axis.
        Adds date to title and saves the plot in vector format (.svg) using a full save path.

        Args:
            date_str (str): Date string in 'YYYY-MM-DD' format.
            save_path (str): Directory or file stem where image should be saved (without extension).
        """
        df = pd.read_csv(self.csv_path)
        steps = df['step']
        
        # Parse date
        date_obj = datetime.strptime(date_str, "%Y-%m-%d")
        short_date = date_obj.strftime("%m-%d")       # e.g. '6 May'
        long_date  = date_obj.strftime("%Y-%m-%d")      # e.g. '2025-05-06'

        fig, host = plt.subplots(figsize=(12, 6))
        fig.subplots_adjust(right=0.75)

        # Additional axes
        par1 = host.twinx()
        par2 = host.twinx()
        par3 = host.twinx()
        par2.spines["right"].set_position(("axes", 1.15))
        par3.spines["right"].set_position(("axes", 1.30))
        for p in [par2, par3]:
            p.spines["right"].set_visible(True)

        # Plot losses
        p0, = host.plot(steps, df['D_loss'], color='C0', label='D_loss')
        p1, = par1.plot(steps, df['G_loss'], color='C1', label='G_loss')
        p2, = par2.plot(steps, df['T_loss'], color='C2', label='T_loss')
        p3, = par3.plot(steps, df['Wasserstein_dist'], color='C3', label='W_dist')

        # Label axes
        host.set_xlabel("Training Step")
        host.set_ylabel("D_loss", color='C0')
        par1.set_ylabel("G_loss", color='C1')
        par2.set_ylabel("T_loss", color='C2')
        par3.set_ylabel("W_dist", color='C3')

        host.tick_params(axis='y', colors='C0')
        par1.tick_params(axis='y', colors='C1')
        par2.tick_params(axis='y', colors='C2')
        par3.tick_params(axis='y', colors='C3')
        
        # scales
        if logscale.get("d"):
            host.set_yscale("log")
        if logscale.get("g"):
            par1.set_yscale("log")
        if logscale.get("t"):
            par2.set_yscale("log")
        if logscale.get("w"):
            par3.set_yscale("log")

        # Title and legend
        lines = [p0, p1, p2, p3]
        host.legend(lines, [l.get_label() for l in lines], loc='upper right')
        host.set_title(f"Training Loss Progression ({short_date})")

        # Save plot in vector format (.svg)
        base_filename = f"{save_path}_loss_plot_{long_date}"
        filename = base_filename + ".svg"
        counter = 1

        # Check if file exists and increment counter until unique name is found
        while os.path.exists(filename):
            filename = f"{base_filename}_{counter}.svg"
            counter += 1

        host.grid(True)
        plt.savefig(filename, format='svg')
        plt.show()

    @classmethod
    def from_checkpoint(cls,
                         ckpt_dir: str,
                         step: int,
                         general: dict,
                         adv_params: dict,
                         G_arch: list,
                         D_arch: list,
                         map_location=None):
        """
        Create a GAN instance and load weights from a given checkpoint step.

        Args:
            ckpt_dir (str): Directory where checkpoints are stored.
            step (int): Checkpoint number (matches save_step).
            general (dict): General training parameters (learning_rate, beta1, beta2, ...).
            adv_params (dict): Adversarial parameters (e.g., N_CRITIC for WGAN-GP).
            G_arch (list): Generator architecture description.
            D_arch (list): Discriminator architecture description.
            map_location: Device mapping for torch.load (default: same as training device).
        Returns:
            GAN: An initialized GAN object with loaded weights.
        """
        # Instantiate fresh GAN
        gan = cls(general, adv_params, G_arch, D_arch)

        # Determine device for loading
        if map_location is None:
            map_location = gan.device

        # Build checkpoint filenames
        netG_file = os.path.join(ckpt_dir, f"netG_step_{step}.pth")
        netD_file = os.path.join(ckpt_dir, f"netD_step_{step}.pth")

        # Load state dicts
        gan.netG.load_state_dict(torch.load(netG_file, map_location=map_location))
        gan.netD.load_state_dict(torch.load(netD_file, map_location=map_location))

        # Optionally, you could also load optimizer states (if saved)
        # gan.optimizerG.load_state_dict(torch.load(os.path.join(ckpt_dir, f"optimG_step_{step}.pth"), map_location))
        # gan.optimizerD.load_state_dict(torch.load(os.path.join(ckpt_dir, f"optimD_step_{step}.pth"), map_location))

        # Update FLAGS or internal counters if needed
        try:
            import FLAGS
            FLAGS.model_step = step
            FLAGS.continue_model = True
        except ImportError:
            pass

        print(f"Loaded GAN checkpoint at step {step} from {ckpt_dir}")
        return gan


C:\Users\MIKEC\.conda\envs\topoenv\lib\site-packages\torchvision\io\image.py:13: UserWarning: Failed to load image Python extension: 
  warn(f"Failed to load image Python extension: {e}")


In [ ]:
""